# Fase 5 · M04: Clasificadores Probabilísticos Simples — VERSIÓN PRUEBA

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 5 — Modelado Clásico |
| **Módulo** | M04 — Clasificadores Probabilísticos Simples |

---

## 🎯 Qué hace

⚡ **Versión prueba** — ejecuta el pipeline completo sobre un subset estratificado del 20% del train. Objetivo: verificar que todo funciona antes de lanzar la versión completa. Sin Optuna, sin serialización definitiva.



## 📋 Requisitos

- `data/05_modelado/X_train_prep.parquet`, `X_test_prep.parquet`
- `data/05_modelado/X_train.parquet`, `X_test.parquet`, `y_train.parquet`, `y_test.parquet`
- `data/05_modelado/pipeline_preprocesamiento.pkl`
- Entorno: `tfm_abandono` (scikit-learn ≥1.3, optuna, imbalanced-learn, codecarbon, joblib)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase5/m04_otros_prueba.html` | Informe HTML versión prueba |





## 🔄 Flujo

```
X_train_prep / y_train  (f5_m01_preparacion)
    ↓ Hiperparámetros por defecto
    ↓ 3-Fold CV estratificado × 9 combinaciones (subset 20%)
    
    ↓ Análisis k óptimo KNN (curva F1 vs k)
    ↓ Curva de aprendizaje (mejor modelo)
    ↓ Calibración de probabilidades + curvas ROC
    
    → HTML prueba orientativo
```

## ➡️ Siguiente

Verificar resultados y lanzar `f5_m04_otros.ipynb`


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN Y DEPENDENCIAS
# ============================================================================

import sys
import warnings
import json
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    StratifiedKFold, StratifiedShuffleSplit,
    cross_validate, learning_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    f1_score, roc_auc_score, precision_score, recall_score,
    accuracy_score, cohen_kappa_score, log_loss,
    confusion_matrix, classification_report, roc_curve
)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline



warnings.filterwarnings('ignore')

# ROOT detection — sube niveles hasta encontrar src/
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(6):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

from src.config import RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.utils.graficos import figura_a_base64
from src.html.render import render_pagina

# Rutas
RUTA_MODELADO = ROOT / 'data' / '05_modelado'


RUTA_HTML_F5  = RUTA_HTML / 'fase5'
crear_directorios([RUTA_HTML_F5])

# Constantes
RANDOM_STATE    = 42
N_SPLITS_CV     = 3   # reducido para velocidad
N_TRIALS_OPTUNA = 20
FAMILIA         = 'otros'
ESTRATEGIAS     = ['none', 'balanced', 'smote']
COLOR           = '#3182ce'
fmt             = formato_numero_es
SUBSET_RATIO    = 0.20  # subset estratificado para versión prueba

info_entorno()
print(f'\n📂 ROOT    : {ROOT}')


print(f'\n⚡ Modo    : PRUEBA — subset {int(SUBSET_RATIO*100)}% del train, sin Optuna')


graficos_b64 = {}  # acumulador de gráficos base64

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGA DE DATOS+ SUBSET ESTRATIFICADO 20%
# ============================================================================
# Lee splits y preprocesados generados por f5_m01_preparacion.
# Los _prep se cargan desde disco directamente — no se re-aplica el transform.
# ============================================================================

X_train = pd.read_parquet(RUTA_MODELADO / 'X_train.parquet')
X_test  = pd.read_parquet(RUTA_MODELADO / 'X_test.parquet')
y_train = pd.read_parquet(RUTA_MODELADO / 'y_train.parquet').squeeze()
y_test  = pd.read_parquet(RUTA_MODELADO / 'y_test.parquet').squeeze()

pipeline_prep = joblib.load(RUTA_MODELADO / 'pipeline_preprocesamiento.pkl')

ruta_train_prep = RUTA_MODELADO / 'X_train_prep.parquet'
ruta_test_prep  = RUTA_MODELADO / 'X_test_prep.parquet'

if ruta_train_prep.exists() and ruta_test_prep.exists():
    X_train_prep = pd.read_parquet(ruta_train_prep)
    X_test_prep  = pd.read_parquet(ruta_test_prep)
    print('✅ Preprocesados cargados desde disco')
else:
    print('⏳ Aplicando pipeline (primera vez)...')
    X_train_prep = pd.DataFrame(
        pipeline_prep.transform(X_train),
        columns=X_train.columns, index=X_train.index
    )
    X_test_prep = pd.DataFrame(
        pipeline_prep.transform(X_test),
        columns=X_test.columns, index=X_test.index
    )
    X_train_prep.to_parquet(ruta_train_prep)
    X_test_prep.to_parquet(ruta_test_prep)
    print('✅ Preprocesados generados y guardados')

FEATURE_NAMES = list(X_train.columns)
N_FEATURES    = len(FEATURE_NAMES)

# Subset estratificado 20% para versión prueba
sss = StratifiedShuffleSplit(n_splits=1, test_size=1-SUBSET_RATIO, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_train_prep = X_train_prep.iloc[idx_sub]
y_train      = y_train.iloc[idx_sub]

print('=' * 55)
print('DATOS CARGADOS  (PRUEBA)')
print('=' * 55)
print(f'Train completo : {fmt(len(X_train))} × {N_FEATURES}')
print(f'Subset prueba  : {fmt(len(X_train_prep))} filas ({int(SUBSET_RATIO*100)}%)')
print(f'X_test  : {fmt(len(X_test))}  × {N_FEATURES}  |  abandono: {y_test.mean()*100:.1f}%')
print(f'Pipeline: {pipeline_prep.__class__.__name__} ✅')
print()
print('⚠️  X_test / y_test INTOCABLES — solo se usan en evaluación final')


✅ Preprocesados cargados desde disco
DATOS CARGADOS  (PRUEBA)
Train completo : 26.896 × 19
Subset prueba  : 5.379 filas (20%)
X_test  : 6.725  × 19  |  abandono: 29.2%
Pipeline: ColumnTransformer ✅

⚠️  X_test / y_test INTOCABLES — solo se usan en evaluación final


In [3]:
# ============================================================================
# CELDA 3: FUNCIONES AUXILIARES
# ============================================================================

CV = StratifiedKFold(n_splits=N_SPLITS_CV, shuffle=True, random_state=RANDOM_STATE)


def construir_pipeline(modelo, estrategia: str):
    """Ensambla modelo con estrategia de balance.
    none     → modelo directo
    balanced → class_weight en el modelo (si lo soporta)
    smote    → SMOTE antes del modelo (ImbPipeline)
    """
    if estrategia == 'smote':
        return ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE)),
            ('model', modelo)
        ])
    return Pipeline([('model', modelo)])


def evaluar_cv(nombre: str, modelo, X, y, estrategia: str) -> dict:
    """5-Fold CV estratificado con métricas completas + tiempo + memoria."""
    pipe = construir_pipeline(modelo, estrategia)
    scoring = {'f1': 'f1', 'roc_auc': 'roc_auc',
               'precision': 'precision', 'recall': 'recall', 'accuracy': 'accuracy'}
    tracemalloc.start()
    t0     = time.time()
    cv_res = cross_validate(pipe, X, y, cv=CV, scoring=scoring,
                            return_train_score=False, n_jobs=-1)
    t_total = time.time() - t0
    _, mem_pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'familia'        : FAMILIA,
        'f1_mean'        : cv_res['test_f1'].mean(),
        'f1_std'         : cv_res['test_f1'].std(),
        'auc_mean'       : cv_res['test_roc_auc'].mean(),
        'auc_std'        : cv_res['test_roc_auc'].std(),
        'precision_mean' : cv_res['test_precision'].mean(),
        'recall_mean'    : cv_res['test_recall'].mean(),
        'accuracy_mean'  : cv_res['test_accuracy'].mean(),
        'tiempo_s'       : round(t_total, 3),
        'memoria_mb'     : round(mem_pico / 1024**2, 2),
    }


def calcular_metricas_test(nombre: str, pipe_fit, X_te, y_te, estrategia: str) -> dict:
    """Métricas completas sobre test para modelo ya entrenado."""
    y_pred  = pipe_fit.predict(X_te)
    y_proba = (
        pipe_fit.predict_proba(X_te)[:, 1]
        if hasattr(pipe_fit, 'predict_proba')
        else pipe_fit.decision_function(X_te)
    )
    if y_proba.min() < 0 or y_proba.max() > 1:
        y_proba = (y_proba - y_proba.min()) / (y_proba.max() - y_proba.min() + 1e-9)
    return {
        'modelo'         : nombre,
        'estrategia'     : estrategia,
        'f1_test'        : round(f1_score(y_te, y_pred), 4),
        'auc_test'       : round(roc_auc_score(y_te, y_proba), 4),
        'precision_test' : round(precision_score(y_te, y_pred), 4),
        'recall_test'    : round(recall_score(y_te, y_pred), 4),
        'accuracy_test'  : round(accuracy_score(y_te, y_pred), 4),
        'kappa_test'     : round(cohen_kappa_score(y_te, y_pred), 4),
        'logloss_test'   : round(log_loss(y_te, y_proba), 4),
    }


print('✅ Funciones auxiliares definidas')
print('   · construir_pipeline  · evaluar_cv')
print('   · calcular_metricas_test')


✅ Funciones auxiliares definidas
   · construir_pipeline  · evaluar_cv
   · calcular_metricas_test


In [4]:
# ============================================================================
# CELDA 4: HIPERPARÁMETROS ÓPTIMOS — KNN · GaussianNB · BernoulliNB
# ============================================================================
# Nota técnica (para el tribunal):
#   KNN, GaussianNB y BernoulliNB son clasificadores de la familia de
#   métodos probabilísticos simples. KNN clasifica por similitud (distancia
#   euclidiana a los k vecinos más próximos); GaussianNB y BernoulliNB
#   estiman probabilidades condicionales bajo el supuesto de independencia
#   de features (Naive Bayes). Son modelos rápidos, interpretables y útiles
#   como baseline antes de pasar a los ensambles (M05+).
#
# Params registrados tras ejecución de Optuna (20 trials × 3 modelos).
# KNN puede ser lento con datasets grandes — se usa n_jobs=-1.
# ⚡ Versión prueba — params por defecto, sin Optuna.
# ============================================================================

FORZAR_OPTUNA  = False
N_TRIALS_M04   = 20

PARAMS_OPTUNA = {
    'KNN': {
        'n_neighbors' : 11,
        'weights'     : 'distance',
        'metric'      : 'euclidean',
        'leaf_size'   : 30,
    },
    'GaussianNB': {
        'var_smoothing': 1e-9,
    },
    'BernoulliNB': {
        'alpha'    : 1.0,
        'binarize' : 0.0,
    },
}
AUC_OPTUNA = {
    'KNN'        : None,
    'GaussianNB' : None,
    'BernoulliNB': None,
}

if True:  # versión prueba — siempre usa params por defecto
    mejores_params = PARAMS_OPTUNA
    print('⚡ Params por defecto (versión prueba)')
    
    for nombre, params in mejores_params.items():
        print(f'   {nombre:<14}: {params}')



⚡ Params por defecto (versión prueba)
   KNN           : {'n_neighbors': 11, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 30}
   GaussianNB    : {'var_smoothing': 1e-09}
   BernoulliNB   : {'alpha': 1.0, 'binarize': 0.0}


In [5]:
# ============================================================================
# CELDA 5: ENTRENAMIENTO — 9 COMBINACIONES (3 modelos × 3 estrategias)
# ============================================================================
# Notas por modelo:
#   KNN         : no tiene class_weight — balance solo via SMOTE
#   GaussianNB  : no tiene class_weight — balance solo via SMOTE
#   BernoulliNB : no tiene class_weight — balance solo via SMOTE
# Los tres modelos disponen de predict_proba nativo — no requieren calibración.
# ⚡ Versión prueba — entrena siempre sobre el subset, sin guardar .pkl ni parquet.
# ============================================================================

MODELOS_M04 = ['KNN', 'GaussianNB', 'BernoulliNB']


def construir_modelo(nombre: str, estrategia: str):
    """Instancia modelo con hiperparámetros por defecto."""
    p = mejores_params[nombre]
    if nombre == 'KNN':
        return KNeighborsClassifier(**p, n_jobs=-1)
    elif nombre == 'GaussianNB':
        return GaussianNB(**p)
    elif nombre == 'BernoulliNB':
        return BernoulliNB(**p)


modelos_fit     = {}
resultados_cv   = []
resultados_test = []

print('=' * 60)
print('ENTRENAMIENTO PRUEBA — 3 modelos × 3 estrategias = 9 combinaciones')
print('=' * 60)
t0_total = time.time()

for nombre in MODELOS_M04:
    print(f'  {nombre}...', end=' ', flush=True)
    for estrategia in ESTRATEGIAS:
        clave  = f'{nombre}__{estrategia}'
        modelo = construir_modelo(nombre, estrategia)
        # CV sobre subset
        res_cv = evaluar_cv(nombre, construir_modelo(nombre, estrategia),
                            X_train_prep, y_train, estrategia)
        resultados_cv.append(res_cv)
        # Entrenamiento final sobre subset
        pipe_final = construir_pipeline(modelo, estrategia)
        pipe_final.fit(X_train_prep, y_train)
        modelos_fit[clave] = pipe_final
        # sin guardar pkl en versión prueba
        # Métricas sobre test completo
        res_test = calcular_metricas_test(nombre, pipe_final,
                                          X_test_prep, y_test, estrategia)
        resultados_test.append(res_test)
    print('✅')

emisiones = None  # sin codecarbon en versión prueba

df_cv   = pd.DataFrame(resultados_cv).sort_values('auc_mean', ascending=False)
df_test = pd.DataFrame(resultados_test)
df_res  = df_cv.merge(df_test, on=['modelo', 'estrategia'])

MEJOR_MODELO     = df_cv.iloc[0]['modelo']
MEJOR_ESTRATEGIA = df_cv.iloc[0]['estrategia']
mejor_clave      = f'{MEJOR_MODELO}__{MEJOR_ESTRATEGIA}'
mejor_pipe       = modelos_fit[mejor_clave]

print(f'\n⏱  Tiempo total: {time.time()-t0_total:.1f}s')
print(f'\n🏆 Mejor (orientativo): {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')


ENTRENAMIENTO PRUEBA — 3 modelos × 3 estrategias = 9 combinaciones
✅ KNN... 
✅ GaussianNB... 
✅ BernoulliNB... 

⏱  Tiempo total: 47.6s

🏆 Mejor (orientativo): KNN + none
   AUC CV = 0.8569 ± 0.0061
   F1  CV = 0.6664 ± 0.0027


In [6]:
# ============================================================================
# CELDA 6: TABLA DE RESULTADOS
# ============================================================================

cols_cv = ['modelo', 'estrategia', 'auc_mean', 'auc_std',
           'f1_mean', 'precision_mean', 'recall_mean', 'tiempo_s']

print('=' * 70)
print('RESULTADOS 5-Fold CV — ordenado por AUC')
print('=' * 70)
print(df_cv[cols_cv].to_string(index=False, float_format='{:.4f}'.format))

if 'auc_test' in df_res.columns:
    cols_test = ['modelo', 'estrategia', 'auc_test', 'f1_test',
                 'precision_test', 'recall_test', 'kappa_test']
    print('\n' + '=' * 70)
    print('MÉTRICAS EN TEST (informativas — selección por CV)')
    print('=' * 70)
    print(df_res[cols_test].sort_values('auc_test', ascending=False)
          .to_string(index=False, float_format='{:.4f}'.format))


RESULTADOS 5-Fold CV — ordenado por AUC
     modelo estrategia  auc_mean  auc_std  f1_mean  precision_mean  recall_mean  tiempo_s
        KNN       none    0.8569   0.0061   0.6664          0.7768       0.5836    4.4900
        KNN   balanced    0.8569   0.0061   0.6664          0.7768       0.5836    5.8780
 GaussianNB      smote    0.8488   0.0052   0.6786          0.6059       0.7711    4.7170
        KNN      smote    0.8476   0.0044   0.6661          0.5786       0.7851    6.0340
 GaussianNB       none    0.8436   0.0085   0.6704          0.6504       0.6917    4.2790
 GaussianNB   balanced    0.8436   0.0085   0.6704          0.6504       0.6917    4.1220
BernoulliNB       none    0.8312   0.0034   0.6350          0.6112       0.6612    7.8530
BernoulliNB   balanced    0.8312   0.0034   0.6350          0.6112       0.6612    4.5890
BernoulliNB      smote    0.8310   0.0030   0.6338          0.5423       0.7635    0.9880

MÉTRICAS EN TEST (informativas — selección por CV)
     mod

In [7]:
# ============================================================================
# CELDA 7: ANÁLISIS K ÓPTIMO — KNN (curva F1 vs k)
# ============================================================================
# KNN es el único modelo de esta familia con un hiperparámetro estructural
# (k = número de vecinos) que afecta directamente al bias-variance tradeoff.
# k pequeño → alta varianza (overfitting); k grande → alto sesgo (underfitting).
# Se grafica F1 CV para k impar entre 1 y 51 para visualizar el codo óptimo.
# ============================================================================


ks    = list(range(1, 52, 2))
f1s_k = []

print('Calculando curva F1 vs k para KNN...', end=' ', flush=True)
t0 = time.time()
for k in ks:
    modelo_k = KNeighborsClassifier(
        n_neighbors=k,
        weights=mejores_params['KNN'].get('weights', 'distance'),
        metric=mejores_params['KNN'].get('metric', 'euclidean'),
        n_jobs=-1
    )
    pipe_k = Pipeline([('model', modelo_k)])
    cv_k   = cross_validate(pipe_k, X_train_prep, y_train, cv=CV,
                             scoring='f1', n_jobs=-1)
    f1s_k.append(cv_k['test_score'].mean())
print(f'✅  ({time.time()-t0:.0f}s)')

k_optimo  = ks[int(np.argmax(f1s_k))]
f1_optimo = max(f1s_k)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ks, f1s_k, 'o-', color=COLOR, linewidth=2, markersize=5)
ax.axvline(k_optimo, color='#e53e3e', linestyle='--', linewidth=1.5,
           label=f'k óptimo = {k_optimo}  (F1={f1_optimo:.4f})')
ax.set_xlabel('Número de vecinos k')
ax.set_ylabel('F1 (5-Fold CV)')
ax.set_title('KNN — F1 en función de k\nBias-variance tradeoff',
             fontsize=11, fontweight='bold')
ax.legend()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['knn_k_optimo'] = figura_a_base64(fig)
plt.close(fig)
print(f'k óptimo: {k_optimo}  |  F1 CV = {f1_optimo:.4f}')


Calculando curva F1 vs k para KNN... ✅  (7s)
k óptimo: 9  |  F1 CV = 0.6675


In [8]:
# ============================================================================
# CELDA 8: CURVA DE APRENDIZAJE
# ============================================================================
# KNN con k pequeño y dataset grande puede ser lento (búsqueda O(n)).
# Se usa submuestra estratificada del 30% del train para mantener tiempos
# razonables — práctica estándar para visualizar tendencia de generalización.
# ============================================================================

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.70, random_state=RANDOM_STATE)
idx_sub, _ = next(sss.split(X_train_prep, y_train))
X_sub = X_train_prep.iloc[idx_sub]
y_sub = y_train.iloc[idx_sub]

print(f'Submuestra: {fmt(len(X_sub))} filas '
      f'({len(X_sub)/len(X_train_prep)*100:.0f}% del train)  '
      f'abandono={y_sub.mean()*100:.1f}%')
print('Calculando curva de aprendizaje...', end=' ', flush=True)
t0 = time.time()

train_sizes, train_scores, cv_scores = learning_curve(
    mejor_pipe, X_sub, y_sub,
    cv=CV, scoring='roc_auc',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
cv_mean    = cv_scores.mean(axis=1)
cv_std     = cv_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train_sizes, train_mean, 'o-', color=COLOR, label='Train AUC', linewidth=2)
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=COLOR)
ax.plot(train_sizes, cv_mean, 'o-', color='#e53e3e', label='CV AUC (5-fold)', linewidth=2)
ax.fill_between(train_sizes, cv_mean - cv_std, cv_mean + cv_std,
                alpha=0.15, color='#e53e3e')
ax.set_xlabel('Tamaño del conjunto de entrenamiento (submuestra 30%)')
ax.set_ylabel('AUC-ROC')
ax.set_title(
    f'Curva de aprendizaje — {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}\n'
    f'Submuestra estratificada 30% del train',
    fontsize=11, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
graficos_b64['curva_aprendizaje'] = figura_a_base64(fig)
plt.close(fig)

gap = train_mean[-1] - cv_mean[-1]
print(f'✅  ({time.time()-t0:.0f}s)')
print(f'Gap train-CV: {gap:.4f}  → '
      f'{"Posible overfitting" if gap > 0.05 else "Generalización correcta"}')


Submuestra: 1.613 filas (30% del train)  abandono=29.3%
✅  (1s)ndo curva de aprendizaje... 
Gap train-CV: 0.1553  → Posible overfitting


In [9]:
# ============================================================================
# CELDA 9: CALIBRACIÓN DE PROBABILIDADES + CURVAS ROC
# ============================================================================
# KNN, GaussianNB y BernoulliNB disponen de predict_proba nativo.
# No requieren CalibratedClassifierCV adicional.
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colores_3 = [COLOR, '#e53e3e', '#38a169']

ax_cal = axes[0]
ax_cal.plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Calibración perfecta')

ax_roc = axes[1]
ax_roc.plot([0, 1], [0, 1], 'k--', linewidth=1)

for idx, nombre_m in enumerate(MODELOS_M04):
    mejor_est = (df_cv[df_cv['modelo'] == nombre_m]
                 .sort_values('auc_mean', ascending=False)
                 .iloc[0]['estrategia'])
    pipe_m = modelos_fit[f'{nombre_m}__{mejor_est}']
    try:
        y_proba = pipe_m.predict_proba(X_test_prep)[:, 1]
    except Exception:
        continue

    frac_pos, mean_pred = calibration_curve(y_test, y_proba, n_bins=10)
    ax_cal.plot(mean_pred, frac_pos, 'o-', color=colores_3[idx],
                label=f'{nombre_m} ({mejor_est})', linewidth=1.8, markersize=5)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc_v = roc_auc_score(y_test, y_proba)
    ax_roc.plot(fpr, tpr, color=colores_3[idx],
                label=f'{nombre_m} AUC={auc_v:.3f}', linewidth=1.8)

ax_cal.set_xlabel('Probabilidad predicha')
ax_cal.set_ylabel('Fracción positivos reales')
ax_cal.set_title('Calibración de probabilidades', fontweight='bold')
ax_cal.legend(fontsize=9)
ax_cal.spines['top'].set_visible(False)
ax_cal.spines['right'].set_visible(False)

ax_roc.set_xlabel('Tasa de falsos positivos')
ax_roc.set_ylabel('Tasa de verdaderos positivos')
ax_roc.set_title('Curvas ROC comparativas', fontweight='bold')
ax_roc.legend(fontsize=9)
ax_roc.spines['top'].set_visible(False)
ax_roc.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['calibracion_roc'] = figura_a_base64(fig)
plt.close(fig)
print('✅ Calibración + ROC generados')


✅ Calibración + ROC generados


In [10]:
# ============================================================================
# CELDA 10: MATRIZ DE CONFUSIÓN + COMPARATIVA AUC
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_pred_mejor = mejor_pipe.predict(X_test_prep)
cm = confusion_matrix(y_test, y_pred_mejor)
ax_cm = axes[0]
im = ax_cm.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax_cm)
clases = ['Continúa (0)', 'Abandona (1)']
ax_cm.set_xticks([0, 1]); ax_cm.set_xticklabels(clases, rotation=30, ha='right')
ax_cm.set_yticks([0, 1]); ax_cm.set_yticklabels(clases)
thresh = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax_cm.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j] > thresh else 'black',
                   fontsize=13, fontweight='bold')
ax_cm.set_title(f'Matriz de confusión\n{MEJOR_MODELO} + {MEJOR_ESTRATEGIA} (test)',
                fontweight='bold')
ax_cm.set_ylabel('Etiqueta real')
ax_cm.set_xlabel('Etiqueta predicha')

ax_bar = axes[1]
aucs_cv     = [df_cv[df_cv['modelo']==m]['auc_mean'].max() for m in MODELOS_M04]
colores_bar = [COLOR if m != MEJOR_MODELO else '#e53e3e' for m in MODELOS_M04]
bars = ax_bar.bar(MODELOS_M04, aucs_cv, color=colores_bar, edgecolor='white')
ymin = max(0.5, min(aucs_cv) - 0.02)
ax_bar.set_ylim(ymin, min(1.0, max(aucs_cv) + 0.02))
ax_bar.set_ylabel('AUC-ROC (mejor estrategia, CV)')
ax_bar.set_title('Comparativa AUC por modelo\n(rojo = mejor)', fontweight='bold')
for bar, val in zip(bars, aucs_cv):
    ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax_bar.spines['top'].set_visible(False)
ax_bar.spines['right'].set_visible(False)

plt.tight_layout()
graficos_b64['cm_comparativa'] = figura_a_base64(fig)
plt.close(fig)

print('✅ Matriz de confusión + comparativa AUC generados')
print()
print('Classification report — mejor modelo en test:')
print(classification_report(y_test, y_pred_mejor,
      target_names=['Continúa', 'Abandona']))


✅ Matriz de confusión + comparativa AUC generados

Classification report — mejor modelo en test:
              precision    recall  f1-score   support

    Continúa       0.85      0.93      0.89      4758
    Abandona       0.78      0.61      0.69      1967

    accuracy                           0.84      6725
   macro avg       0.82      0.77      0.79      6725
weighted avg       0.83      0.84      0.83      6725



In [11]:
# ============================================================================
# CELDA 12: GENERAR HTML
# ============================================================================
# Texto generado automáticamente desde resultados en memoria.
# KPIs, tablas y gráficos idénticos al patrón de m02/m03.
# ============================================================================

RUTA_HTML_SALIDA = RUTA_HTML_F5 / 'm04_otros_prueba.html'

aviso_prueba = '''
<div style="background:#fff3cd;border:1px solid #ffc107;border-radius:8px;
            padding:14px 18px;margin-bottom:20px;">
  <strong>⚡ Versión prueba</strong> — Subset 20% del train · 3-Fold CV · Sin Optuna<br>
  Los resultados son orientativos. Ejecutar <code>f5_m04_otros.ipynb</code> para los artefactos definitivos.
</div>'''

# --- KPIs ---
kpis = [
    {'valor': '3',                                 'titulo': 'Modelos'},
    {'valor': '3',                                 'titulo': 'Estrategias'},
    {'valor': '9',                                 'titulo': 'Combinaciones'},
    {'valor': str(N_TRIALS_M04),                   'titulo': 'Trials Optuna'},
    {'valor': f'{df_cv.iloc[0]["auc_mean"]:.3f}', 'titulo': 'Mejor AUC CV'},
    {'valor': f'{df_cv.iloc[0]["f1_mean"]:.3f}',  'titulo': 'Mejor F1 CV'},
]
kpis_html = '<div style="display:flex;flex-wrap:wrap;gap:16px;margin:16px 0;">' + ''.join(
    f'<div style="background:#f7fafc;border:1px solid #e2e8f0;border-radius:10px;'
    f'padding:18px 28px;text-align:center;min-width:120px;">'
    f'<div style="font-size:2rem;font-weight:700;color:#3182ce;">{k["valor"]}</div>'
    f'<div style="font-size:0.85rem;color:#555;margin-top:4px;">{k["titulo"]}</div>'
    f'</div>'
    for k in kpis
) + '</div>'

# --- Texto dinámico ---
segundo   = df_cv.iloc[1]
diff_auc  = df_cv.iloc[0]['auc_mean'] - segundo['auc_mean']
diff_f1   = df_cv.iloc[0]['f1_mean']  - segundo['f1_mean']
peor      = df_cv.iloc[-1]
rango_auc = df_cv.iloc[0]['auc_mean'] - peor['auc_mean']

texto_dinamico = (
    f'<p>El análisis de <strong>9 combinaciones</strong> '
    f'(3 modelos × 3 estrategias de balance) revela que '
    f'<strong>{MEJOR_MODELO}</strong> con estrategia <strong>{MEJOR_ESTRATEGIA}</strong> '
    f'es la combinación ganadora de la familia probabilística simple, alcanzando un '
    f'<strong>AUC CV de {df_cv.iloc[0]["auc_mean"]:.4f} '
    f'± {df_cv.iloc[0]["auc_std"]:.4f}</strong> '
    f'y un <strong>F1 de {df_cv.iloc[0]["f1_mean"]:.4f}</strong>.</p>'
    f'<p>Supera al segundo clasificador ({segundo["modelo"]} + {segundo["estrategia"]}) en '
    f'<strong>{diff_auc:.4f} puntos de AUC</strong> y <strong>{diff_f1:.4f} de F1</strong>. '
    f'El rango total entre mejor y peor combinación es de {rango_auc:.4f} puntos de AUC.</p>'
    f'<p>Los tres modelos disponen de <code>predict_proba</code> nativo, '
    f'lo que garantiza probabilidades calibradas sin calibración adicional. '
    f'KNN es el más sensible al parámetro k y a la escala de las features '
    f'(requiere StandardScaler del pipeline); '
    f'GaussianNB y BernoulliNB son prácticamente instantáneos en entrenamiento.</p>'
)

# --- Tabla pivotada ---
filas_pivot = ''
for modelo in MODELOS_M04:
    sub      = df_cv[df_cv['modelo'] == modelo].sort_values('auc_mean', ascending=False).iloc[0]
    es_mejor = modelo == MEJOR_MODELO
    bg       = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    estrella = ' 🏆' if es_mejor else ''
    auc_opt_str = f'{AUC_OPTUNA[modelo]:.4f}' if AUC_OPTUNA[modelo] is not None else '—'
    filas_pivot += (
        f'<tr style="{bg}">'
        f'<td>{modelo}{estrella}</td>'
        f'<td>{sub["estrategia"]}</td>'
        f'<td>{sub["auc_mean"]:.4f} &plusmn; {sub["auc_std"]:.4f}</td>'
        f'<td>{sub["f1_mean"]:.4f}</td>'
        f'<td>{sub["precision_mean"]:.4f}</td>'
        f'<td>{sub["recall_mean"]:.4f}</td>'
        f'<td>{auc_opt_str}</td>'
        f'<td><code>{json.dumps(PARAMS_OPTUNA[modelo])}</code></td>'
        f'</tr>'
    )
tabla_pivot = (
    '<p>Una fila por modelo con su mejor estrategia. 🏆 = mejor combinación global.</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Mejor estrategia</th>'
    '<th>AUC CV (mean&plusmn;std)</th><th>F1 CV</th>'
    '<th>Precisión</th><th>Recall</th>'
    '<th>AUC Optuna</th><th>Hiperparámetros</th>'
    f'</tr></thead><tbody>{filas_pivot}</tbody></table>'
)

# --- Tabla completa CV ---
filas_cv = ''
for _, r in df_cv.iterrows():
    es_mejor = r['modelo'] == MEJOR_MODELO and r['estrategia'] == MEJOR_ESTRATEGIA
    bg = 'background:#ebf8ff;font-weight:600;' if es_mejor else ''
    filas_cv += (
        f'<tr style="{bg}">'
        f'<td>{r["modelo"]}</td><td>{r["estrategia"]}</td>'
        f'<td>{r["auc_mean"]:.4f} &plusmn; {r["auc_std"]:.4f}</td>'
        f'<td>{r["f1_mean"]:.4f}</td>'
        f'<td>{r["precision_mean"]:.4f}</td>'
        f'<td>{r["recall_mean"]:.4f}</td>'
        f'<td>{r["tiempo_s"]:.1f}s</td></tr>'
    )
tabla_cv = (
    f'<p>Ordenado por AUC-ROC descendente. Fila destacada = mejor combinación: '
    f'<strong>{MEJOR_MODELO} + {MEJOR_ESTRATEGIA}</strong> '
    f'(AUC={df_cv.iloc[0]["auc_mean"]:.4f}).</p>'
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Estrategia</th><th>AUC (mean&plusmn;std)</th>'
    '<th>F1</th><th>Precisión</th><th>Recall</th><th>Tiempo</th>'
    f'</tr></thead><tbody>{filas_cv}</tbody></table>'
)

# --- Hiperparámetros Optuna ---
filas_params_list = []
for m in MODELOS_M04:
    auc_str = f'{AUC_OPTUNA[m]:.4f}' if AUC_OPTUNA[m] is not None else '—'
    filas_params_list.append(
        f'<tr><td>{m}</td><td><code>{json.dumps(PARAMS_OPTUNA[m])}</code></td>'
        f'<td>{auc_str}</td></tr>'
    )
tabla_params = (
    '<table class="tabla-datos"><thead><tr>'
    '<th>Modelo</th><th>Hiperparámetros óptimos (Optuna)</th><th>AUC búsqueda</th>'
    f'</tr></thead><tbody>{"".join(filas_params_list)}</tbody></table>'
)

# --- CO2 ---
co2_html = (
    f'<p>🌿 Huella CO2 del entrenamiento: <strong>{emisiones:.6f} kg CO2eq</strong></p>'
    if False
    else '<p>ℹ️ Modelos cargados desde disco — sin nuevo entrenamiento en esta ejecución.</p>'
)

def img_tag(key):
    b64 = graficos_b64.get(key)
    if not b64:
        return '<p><em>Gráfico no disponible</em></p>'
    return f'<img src="data:image/png;base64,{b64}" style="max-width:100%;border-radius:8px;">'

secciones = (
    '<section class="seccion"><h2>Resumen</h2>' + aviso_prueba + kpis_html + '</section>'
    '<section class="seccion"><h2>Conclusiones del análisis</h2>' + texto_dinamico + '</section>'
    '<section class="seccion"><h2>Comparativa por modelo — mejor estrategia de cada uno</h2>' + tabla_pivot + '</section>'
    '<section class="seccion"><h2>Resultados completos 5-Fold CV — 9 combinaciones</h2>' + tabla_cv + '</section>'
    f'<section class="seccion"><h2>Hiperparámetros óptimos — Optuna ({N_TRIALS_M04} trials × 3 modelos)</h2>' + tabla_params + '</section>'
    '<section class="seccion"><h2>KNN — análisis k óptimo (bias-variance tradeoff)</h2>'
    '<p>Curva F1 en función del número de vecinos k. El codo óptimo marca el mejor equilibrio entre varianza (k pequeño) y sesgo (k grande).</p>'
    + img_tag('knn_k_optimo') + '</section>'
    f'<section class="seccion"><h2>Curva de aprendizaje — {MEJOR_MODELO}</h2>'
    '<p>Submuestra estratificada 30% del train. Convergencia entre train y CV indica buena generalización.</p>'
    + img_tag('curva_aprendizaje') + '</section>'
    '<section class="seccion"><h2>Calibración de probabilidades y curvas ROC</h2>'
    '<p>Los tres modelos exponen predict_proba nativo — las probabilidades son directamente interpretables como riesgo de abandono.</p>'
    + img_tag('calibracion_roc') + '</section>'
    '<section class="seccion"><h2>Matriz de confusión y comparativa AUC</h2>'
    + img_tag('cm_comparativa') + '</section>'
    '<section class="seccion"><h2>Sostenibilidad computacional</h2>' + co2_html + '</section>'
)

render_pagina(
    'f5_m04_otros_prueba.ipynb',
    secciones,
    RUTA_HTML_SALIDA,
    carpeta_notebook='fase5_modelado'
)
print(f'\n✅ HTML generado: {RUTA_HTML_SALIDA}')



✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase5\m04_otros_prueba.html


In [12]:
# ============================================================================
# CELDA 13: RESUMEN FINAL
# ============================================================================

print('=' * 60)
print('RESUMEN FINAL — f5_m04_otros_prueba')
print('=' * 60)
print(f'Subset      : {fmt(len(X_train_prep))} filas ({int(SUBSET_RATIO*100)}% train)')
print(f'Familia     : Clasificadores probabilísticos simples')
print(f'Modelos     : KNN · GaussianNB · BernoulliNB')
print(f'Estrategias : none · balanced · smote  (9 combinaciones)')
print(f'CV folds    : {N_SPLITS_CV}  (reducido)')
print( 'Optuna      : ❌ No ejecutado (params por defecto)')
print()
print(f'🏆 Mejor  (orientativo): {MEJOR_MODELO} + {MEJOR_ESTRATEGIA}')
print(f'   AUC CV = {df_cv.iloc[0]["auc_mean"]:.4f} ± {df_cv.iloc[0]["auc_std"]:.4f}')
print(f'   F1  CV = {df_cv.iloc[0]["f1_mean"]:.4f} ± {df_cv.iloc[0]["f1_std"]:.4f}')
print()
print('✅ Pipeline verificado — listo para lanzar f5_m04_otros.ipynb')


RESUMEN FINAL — f5_m04_otros_prueba
Subset      : 5.379 filas (20% train)
Familia     : Clasificadores probabilísticos simples
Modelos     : KNN · GaussianNB · BernoulliNB
Estrategias : none · balanced · smote  (9 combinaciones)
CV folds    : 3  (reducido)
Optuna      : ❌ No ejecutado (params por defecto)

🏆 Mejor  (orientativo): KNN + none
   AUC CV = 0.8569 ± 0.0061
   F1  CV = 0.6664 ± 0.0027

✅ Pipeline verificado — listo para lanzar f5_m04_otros.ipynb
